In [0]:
import json
import os
from urllib.parse import urljoin
import pandas as pd
import requests
BASE_URL = 'https://cvijan.github.io/nss-api/v2/'
FILES_URL = 'https://cvijan.github.io/nss-api/files/'
RAW_DIR = 'data/raw'
PROCESSED_DIR = 'data/processed'

In [0]:
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

In [0]:
# 1. Позив API-ја. Написати функцију get_json(url: str) -> dict која шаље GET захтев.
# Подиже грешку ако статус није успешан и враћа одговор као речник ако јесте.
# get_json(BASE_URL + 'status.json')['service'] враћа 'Northern Star Shop
# API'
# get_json(BASE_URL + 'orders/page-99.json') подиже requests.HTTPError


def get_json(url: str) -> dict:
    response = requests.get(url)
    response.raise_for_status()
    
    return response.json()

print(get_json(BASE_URL + 'status.json')['service'])
# 'Northern Star Shop API'

#print(get_json(BASE_URL + 'orders/page-99.json'))

# HTTPError: 404 Client Error: Not Found for url: https://cvijan.github.io/nss-api/v2/orders/page-99.json


Northern Star Shop API


In [0]:
# 2. Странице. Написати функцију fetch_all(first_url: str) -> list која прати next
# линкове и враћа записе са свих страна у једној листи
# Помоћ: urljoin(url, page['next'])
# Input: BASE_URL + 'orders/page-1.json'' даје 1852 записа


def fetch_all(first_url: str) -> list:
    records = []
    url = first_url

    while url:
        page = get_json(url)
        records.extend(page['data'])
        if page['next']:
            url = urljoin(url, page['next'])
        else:
            url = None
    
    return records

orders = fetch_all(BASE_URL + 'orders/page-1.json')
len(orders) # 1852


1852

In [0]:
# 3. Чување сирових података. Написати функцију save_json(records: list, path: str) ->
# None која листу речника уписује у JSON фајл без икаквих измена.
# json.dump(records, f, ensure_ascii=False, indent=2)
# save_json(orders, os.path.join(RAW_DIR, 'orders.json')) -> прави се фајл
# data/raw/orders.json

def save_json(records: list, path:str) -> None:
    with open(path, 'w', encoding= 'utf-8') as f:
        json.dump(records, f, ensure_ascii= False)

save_json(orders, os.path.join(RAW_DIR, 'orders.json'))
# napravio je fajl data/raw/orders.json

In [0]:
# 4. Preuzimanje fajla, kopiraj bez izmena
def download_file(url: str, path: str) -> int:
    """Download a file and save it unchanged. Returns its size in bytes."""
    response = requests.get(url, timeout=30)
    response.raise_for_status() # 404 and other errors stop here

    with open(path, 'wb') as f: # 'wb' = write bytes, needed for Parquet
        f.write(response.content) # .content = the file's bytes,unchanged

    return len(response.content)

download_file(FILES_URL + 'teams.csv', os.path.join(RAW_DIR,
'teams.csv')) 
#801 u data/raw pojavljuje se teams.csv

801

In [0]:
# 5. Написати функцију extract(raw_dir: str) -> dict која преузима сва четири извора у
# raw_dir и враћа речник са четири DataFrame-а: 'orders', 'products', 'teams',
# 'matches'
# 1. Поруџбине читамо помоћу API-ја (fetch_all) (orders.json)
# 2. Фајлови products.csv, teams.csv, matches.parquet се добијају помоћу функције download_file
# 3. Поруџбине у DataFrame: pd.json_normalize(orders, sep='_')
# Након тога преименовати колоне
# customer_name у customer
# customer_city у city
# 4. Остало читамо из raw_dir (где смо у кораку 2 скинули податке)
# По потреби, pd.read_csv(...) или pd.read_parquet(...)
# 5. Провера: orders (1852, 8), products (108, 7), teams (20, 5), matches (120, 9)


def extract(raw_dir: str) -> dict:
    # Porudzbine
    orders = fetch_all(BASE_URL + 'orders/page-1.json')
    save_json(orders, os.path.join(RAW_DIR, 'orders.json'))
    
    # Skidanje fajlova
    download_file(FILES_URL + 'products.csv', 
                  os.path.join(RAW_DIR, 'products.csv'))

    download_file(FILES_URL + 'teams.csv', 
                  os.path.join(RAW_DIR, 'teams.csv'))

    download_file(FILES_URL + 'matches.parquet', 
                  os.path.join(RAW_DIR, 'matches.parquet'))

    # Pretvaranje porudzbina u DataFrame
    orders_df = pd.json_normalize(orders, sep='_')

    # Preimenovanje kolona
    orders_df = orders_df.rename(columns={
        'customer_name': 'customer',
        'customer_city': 'city'
    })

    # Citanje fajlova
    products_df = pd.read_csv(os.path.join(RAW_DIR, 'products.csv'))
    teams_df = pd.read_csv(os.path.join(RAW_DIR, 'teams.csv'))
    matches_df = pd.read_parquet(os.path.join(RAW_DIR, 'matches.parquet'))


    return {
        'orders': orders_df,
        'products': products_df,
        'teams': teams_df,
        'matches': matches_df
    }



data = extract(RAW_DIR)

for name,df in data.items():
    print(name, df.shape)

# orders (1852, 8)
# products (108, 7)
# teams (20, 5)
# matches (120, 9)


orders (1852, 8)
products (108, 7)
teams (20, 5)
matches (120, 9)


In [0]:
# 6. Производи. Написати функцију clean_products(df: pd.DataFrame) -> pd.DataFrame
# која чисти каталог производа
# Ако се понавља исти производ, садржати први ред за сваки product_id
# team_code треба да буде без размака, има велика слова (npm -> NPM), празан ред остаје празан (то су производи лиге, без тима)
# category треба да буде без размака, има мала слова
# price је текст у три облика '1490.00', '1490 RSD', '1490,00' а треба да постане
# само float (у сваком случају)
# Помоћ: .str.replace(' RSD', '', regex=False), pd.to_numeric(...)
# active трансформишемо у True/False ('yes', 'Yes', 'YES' постају True), све остало
# постаје False
# Output: 105 редова, price је типа float64, active је True за 101 производ
# Ред 1003 изгледа овако: {'team_code': 'NPM', 'category': 'scarves',
# 'price': 1490.0, 'active': True, ...}

def clean_products(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    # Uklanja ponovljene proizvode
    df = df.drop_duplicates(subset='product_id', keep='first')

    # Ciscenje team_code
    df['team_code'] = (
        df['team_code']
        .fillna('')
        .astype(str)
        .str.strip()
        .str.upper()
    )
    # Ciscenje category

    df['category'] = (
        df['category']
        .astype(str)
        .str.strip()
        .str.lower()
    )
    # Ciscenje price

    df['price'] = (
        df['price']
        .astype(str)
        .str.strip()
        .str.lower()
        .str.replace(' RSD', '', regex=False)
        .str.replace(' ,', '.', regex=False)
        
    )
    df['price'] = pd.to_numeric(df['price'], errors='coerce')

    # Ciscenje active

    df['active'] = (
        df['active']
        .astype(str)
        .str.strip()
        .str.lower()
        .eq('yes')
    )


    return df

products_clean = clean_products(data['products'])

print(products_clean)


     product_id           sku       product_name team_code     category  \
0          1001  NPM-JER-1001    NPM Home Jersey       NPM      jerseys   
1          1002  NPM-JER-1002    NPM Away Jersey       NPM      jerseys   
2          1003  NPM-SCA-1003          NPM Scarf       NPM      scarves   
3          1004  NPM-CAP-1004            NPM Cap       NPM         caps   
4          1005  NPM-ACC-1005            NPM Mug       NPM  accessories   
..          ...           ...                ...       ...          ...   
100        1101  NSL-BAL-1101     NSL Match Ball                  balls   
101        1102  NSL-BAL-1102  NSL Training Ball                  balls   
102        1103  NSL-ACC-1103   NSL Water Bottle            accessories   
103        1104  NSL-ACC-1104       NSL Backpack            accessories   
104        1105  NSL-ACC-1105       NSL Keychain            accessories   

      price  active  
0    7490.0    True  
1    6990.0    True  
2       NaN    True  
3       NaN

In [0]:
# 7. Поруџбине. Написати функцију clean_orders(df: pd.DataFrame) -> pd.DataFrame
# order_id, customer, city, payment без размака;
# order_id велика слова
# customer и city -> title()
# Град који недостаје (isna) -> 'Unknown'


def clean_orders(df: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()


    df['order_id'] = (
        df['order_id']
        .astype(str)
        .str.strip()
        .str.upper()
    )

    df['customer'] = (
        df['customer']
        .astype(str)
        .str.strip()
        .str.title()
    )

    df['city'] = (
        df['city']
        .fillna('Unknown')
        .astype(str)
        .str.strip()
        .str.title()
    )

    df['payment'] = (
        df['payment']
        .astype(str)
        .str.strip()
    )

    return df

orders_clean = clean_orders(data['orders'])

print("Broj porudžbina:", len(orders_clean))
print("Prvih 5 redova:")
print(orders_clean.head())
print("Nedostajući gradovi:", orders_clean['city'].isna().sum())
print("Unknown gradovi:", (orders_clean['city'] == 'Unknown').sum())

Broj porudžbina: 1852
Prvih 5 redova:
    order_id  order_date  product_id quantity unit_price payment  \
0  NSS-00001  2026-08-15        1002        1     6990.0    cash   
1  NSS-00002  2026-08-15        1094        3     1290.0    card   
2  NSS-00003  2026-08-15        1041        1     6990.0    card   
3  NSS-00004  2026-08-15        1075        1      990.0    cash   
4  NSS-00005  2026-08-15        1044        1     1290.0    card   

            customer       city  
0  Nikola Stojanović  Zrenjanin  
1    Jovana Pavlović      Čačak  
2      Sara Živković    Beograd  
3      Sara Đorđević   Subotica  
4     Stefan Popović        Niš  
Nedostajući gradovi: 0
Unknown gradovi: 12


In [0]:
# 8. Дупликати. Написати функцију remove_duplicate_orders(df: pd.DataFrame) -> tuple
# која враћа (unique, duplicates) по order_id, задржава први
# Output: 1840 јединствених, 12 дупликата

def remove_duplicate_orders(df: pd.DataFrame) -> tuple:
    
    unique = df.drop_duplicates(subset='order_id',keep='first').copy()

    duplicates = df[df.duplicated(subset='order_id',keep='first')].copy()

    return unique, duplicates

unique, duplicates = remove_duplicate_orders(orders_clean)

print("Jedinstvene porudžbine:", len(unique))
print("Duplikati:", len(duplicates))



Jedinstvene porudžbine: 1840
Duplikati: 12


In [0]:

# 9.Validacija podataka
def validate_order(row: dict, product_ids: set) -> str:
    if not str(row.get('order_id', '')).startswith('NSS-'):
        return 'invalid_order'

    quantity = pd.to_numeric(row.get('quantity'), errors='coerce')

    if pd.isna(quantity):
        return 'missing_quantity'

    if quantity <= 0 or quantity % 1 != 0:
        return 'invalid_quantity'

    unit_price = pd.to_numeric(row.get('unit_price'), errors='coerce')

    if pd.isna(unit_price) or unit_price <= 0:
        return 'invalid_price'

    if row.get('product_id') not in product_ids:
        return 'unknown_product'

    return 'OK'

product_ids = set(products_clean['product_id'])

print(validate_order({
    'order_id': 'NSS-123',
    'quantity': 2,
    'unit_price': 1490,
    'product_id': products_clean.iloc[0]['product_id']
}, product_ids))


OK


In [0]:

# 10. Transformacija
def transform_orders(df: pd.DataFrame, products: pd.DataFrame) -> pd.DataFrame:
    df = df.copy()

    df['quantity'] = pd.to_numeric(df['quantity']).astype(int)
    df['unit_price'] = pd.to_numeric(df['unit_price']).astype(float)
    df['order_date'] = pd.to_datetime(df['order_date'])

    df['total'] = df['quantity'] * df['unit_price']

    product_info = products[
        ['product_id', 'product_name', 'team_code', 'category']
    ]

    df = df.merge(product_info, on='product_id', how='left')

    return df

transformed = transform_orders(valid, products_clean)

print("Broj redova:", len(transformed))
print("Broj kolona:", len(transformed.columns))
print("Kolone:", transformed.columns.tolist())
print(transformed.head())

Broj redova: 1800
Broj kolona: 15
Kolone: ['order_id', 'order_date', 'product_id', 'quantity', 'unit_price', 'payment', 'customer', 'city', 'total', 'product_name_x', 'team_code_x', 'category_x', 'product_name_y', 'team_code_y', 'category_y']
    order_id order_date  product_id  quantity  unit_price payment  \
0  NSS-00001 2026-08-15        1002         1      6990.0    cash   
1  NSS-00002 2026-08-15        1094         3      1290.0    card   
2  NSS-00003 2026-08-15        1041         1      6990.0    card   
3  NSS-00004 2026-08-15        1075         1       990.0    cash   
4  NSS-00005 2026-08-15        1044         1      1290.0    card   

            customer       city   total   product_name_x team_code_x  \
0  Nikola Stojanović  Zrenjanin  6990.0  NPM Away Jersey         NPM   
1    Jovana Pavlović      Čačak  3870.0          BRU Cap         BRU   
2      Sara Živković    Beograd  6990.0  RVA Home Jersey         RVA   
3      Sara Đorđević   Subotica   990.0          LSB M

In [0]:

# 11.
def split_orders(df: pd.DataFrame, products: pd.DataFrame) -> tuple:
    cleaned = clean_orders(df)

    unique, duplicates = remove_duplicate_orders(cleaned)

    product_ids = set(products['product_id'])

    valid_rows = []
    rejected_rows = []

    for _, row in unique.iterrows():
        record = row.to_dict()

        reason = validate_order(record, product_ids)

        if reason == 'OK':
            valid_rows.append(record)
        else:
            record['reason'] = reason
            rejected_rows.append(record)

    for _, row in duplicates.iterrows():
        record = row.to_dict()
        record['reason'] = 'duplicate'
        rejected_rows.append(record)

    valid = pd.DataFrame(valid_rows)
    rejected = pd.DataFrame(rejected_rows)

    valid = transform_orders(valid, products)

    return valid, rejected

valid, rejected = split_orders(data['orders'], products_clean)

print("Ispravne porudžbine:", len(valid))
print("Odbačene porudžbine:", len(rejected))
print("Duplikati:", (rejected['reason'] == 'duplicate').sum())


Ispravne porudžbine: 1800
Odbačene porudžbine: 52
Duplikati: 12


In [0]:
# 12. Rezultati timova
def team_results(matches: pd.DataFrame, teams: pd.DataFrame) -> pd.DataFrame:
    played = matches[matches['status'] == 'finished'].copy()

    played['kickoff_ts'] = pd.to_datetime(played['kickoff_ts'])

    codes = dict(zip(teams['team_id'], teams['code']))

    home = pd.DataFrame({
        'match_id': played['match_id'],
        'match_date': played['kickoff_ts'].dt.normalize(),
        'team_code': played['home_team_id'].map(codes),
        'opponent_code': played['away_team_id'].map(codes),
        'goals_for': played['home_goals'],
        'goals_against': played['away_goals']
    })

    away = pd.DataFrame({
        'match_id': played['match_id'],
        'match_date': played['kickoff_ts'].dt.normalize(),
        'team_code': played['away_team_id'].map(codes),
        'opponent_code': played['home_team_id'].map(codes),
        'goals_for': played['away_goals'],
        'goals_against': played['home_goals']
    })

    results = pd.concat([home, away], ignore_index=True)

    results['result'] = 'D'

    results.loc[
        results['goals_for'] > results['goals_against'], 'result'
    ] = 'W'

    results.loc[
        results['goals_for'] < results['goals_against'], 'result'
    ] = 'L'

    results['points'] = results['result'].map({
        'W': 3,
        'D': 1,
        'L': 0
    })

    return results


results = team_results(data['matches'], data['teams'])

print("Broj redova:", len(results))
print("Broj utakmica:", results['match_id'].nunique())
print("Kolone:", results.columns.tolist())
print(results.head())


Broj redova: 238
Broj utakmica: 119
Kolone: ['match_id', 'match_date', 'team_code', 'opponent_code', 'goals_for', 'goals_against', 'result', 'points']
   match_id match_date team_code opponent_code  goals_for  goals_against  \
0   2627001 2026-08-15       BRU           FWR          2              1   
1   2627002 2026-08-15       CHS           VDC          0              1   
2   2627003 2026-08-15       MBA           WFT          3              0   
3   2627004 2026-08-15       NPM           IVU          1              0   
4   2627005 2026-08-15       PCU           HGA          1              1   

  result  points  
0      W       3  
1      L       0  
2      W       3  
3      W       3  
4      D       1  


In [0]:
# 13. Tabela lige
def league_table(results: pd.DataFrame) -> pd.DataFrame:
    table = results.groupby('team_code').agg(
        played=('match_id', 'count'),
        won=('result', lambda x: (x == 'W').sum()),
        drawn=('result', lambda x: (x == 'D').sum()),
        lost=('result', lambda x: (x == 'L').sum()),
        goals_for=('goals_for', 'sum'),
        goals_against=('goals_against', 'sum'),
        points=('points', 'sum')
    ).reset_index()

    table['goal_diff'] = table['goals_for'] - table['goals_against']

    table = table.sort_values(
        by=['points', 'goal_diff', 'goals_for'],
        ascending=[False, False, False]
    ).reset_index(drop=True)

    table.insert(0, 'position', range(1, len(table) + 1))

    return table

league = league_table(results)

print("Broj timova:", len(league))
print("Prva dva tima:")
print(league.head(2)[['position', 'team_code', 'points']])

print("HGA i OKT:")
print(league[league['team_code'].isin(['HGA', 'OKT'])][['team_code', 'played']])

Broj timova: 20
Prva dva tima:
   position team_code  points
0         1       FWR      26
1         2       BMC      23
HGA i OKT:
   team_code  played
4        HGA      11
16       OKT      11


In [0]:
# 14. Match windows
def match_windows(results: pd.DataFrame) -> pd.DataFrame:
    windows_list = []

    for d in range(4):
        temp = results[['team_code', 'match_date', 'result']].copy()

        temp['date'] = temp['match_date'] + pd.Timedelta(days=d)

        windows_list.append(temp)

    windows = pd.concat(windows_list, ignore_index=True)

    windows = windows[
        ['team_code', 'date', 'result']
    ].rename(columns={'result': 'window'})

    return windows

windows = match_windows(results)

print("Broj redova:", len(windows))
print("Kolone:", windows.columns.tolist())

print(windows[
    (windows['team_code'] == 'BRU') &
    (windows['date'] >= '2026-08-15') &
    (windows['date'] <= '2026-08-18')
])

Broj redova: 952
Kolone: ['team_code', 'date', 'window']
    team_code       date window
0         BRU 2026-08-15      W
238       BRU 2026-08-16      W
476       BRU 2026-08-17      W
714       BRU 2026-08-18      W


In [0]:
# 15. Revenue
def revenue_by(df: pd.DataFrame, column: str) -> pd.DataFrame:
    revenue = df.groupby(column)['total'].sum().reset_index(
        name='revenue'
    )

    revenue = revenue.sort_values(
        by='revenue',
        ascending=False
    ).reset_index(drop=True)

    return revenue


category_revenue = revenue_by(valid, 'category')

print("Najprodavanija kategorija:")
print(category_revenue.iloc[0])

Najprodavanija kategorija:
category      jerseys
revenue     7094410.0
Name: 0, dtype: object


In [0]:
# 16. Sales after results
def sales_after_results(
    valid: pd.DataFrame,
    windows: pd.DataFrame
) -> pd.DataFrame:

    # 1. Pravimo kalendar – svaki tim x svaki dan
    teams = pd.DataFrame({
        'team_code': windows['team_code'].unique()
    })

    days = pd.DataFrame({
        'order_date': pd.to_datetime(valid['order_date']).dt.normalize().unique()
    })

    calendar = teams.merge(days, how='cross')

    # 2. Dnevni promet po timu i datumu
    daily_revenue = valid.groupby(
        ['team_code', 'order_date']
    )['total'].sum().reset_index(name='revenue')

    daily_revenue['order_date'] = pd.to_datetime(
        daily_revenue['order_date']
    ).dt.normalize()

    calendar = calendar.merge(
        daily_revenue,
        on=['team_code', 'order_date'],
        how='left'
    )

    calendar['revenue'] = calendar['revenue'].fillna(0)

    # 3. Povezujemo kalendar sa rezultatima utakmica
    windows = windows.copy()
    windows['date'] = pd.to_datetime(windows['date']).dt.normalize()

    calendar = calendar.merge(
        windows,
        left_on=['team_code', 'order_date'],
        right_on=['team_code', 'date'],
        how='left'
    )

    # 4. Dani koji nisu u prozoru utakmice
    calendar['window'] = calendar['window'].fillna('none')

    # 5. Grupisanje po rezultatu
    effect = calendar.groupby('window').agg(
        team_days=('team_code', 'count'),
        revenue=('revenue', 'sum'),
        avg_daily_revenue=('revenue', 'mean')
    ).reset_index()

    effect['avg_daily_revenue'] = effect['avg_daily_revenue'].round(2)

    effect = effect.sort_values(
        by='avg_daily_revenue',
        ascending=False
    ).reset_index(drop=True)

    return effect

effect = sales_after_results(valid, windows)

print("Prodaja nakon rezultata utakmica:")
print(effect)


Prodaja nakon rezultata utakmica:
  window  team_days    revenue  avg_daily_revenue
0      W        339  2999230.0            8847.29
1      D        228  1180180.0            5176.23
2   none        954  4427150.0            4640.62
3      L        339  1012330.0            2986.22


In [0]:
# 17. Load
def load(outputs: dict, processed_dir: str) -> dict:
    written = {}

    for name, df in outputs.items():
        path = os.path.join(processed_dir, name)

        if name.endswith('.parquet'):
            df.to_parquet(path, index=False)
        else:
            df.to_csv(path, index=False)

        written[name] = len(df)

    return written

In [0]:

def run_pipeline(raw_dir: str, processed_dir: str) -> dict:
    data = extract(raw_dir)
    products = clean_products(data['products'])
    valid, rejected = split_orders(data['orders'], products)
    results = team_results(data['matches'], data['teams'])
    windows = match_windows(results)
    sales_effect = sales_after_results(valid, windows)
    outputs = {
        'clean_orders.parquet': valid,
        'rejected_orders.csv': rejected,
        'products.csv': products,
        'league_table.csv': league_table(results),
        'revenue_by_team.csv': revenue_by(
            valid.dropna(subset=['team_code']),
            'team_code'
        ),
        'revenue_by_category.csv': revenue_by(
            valid,
            'category'
        ),
        'sales_after_results.csv': sales_effect
    }
    written = load(outputs, processed_dir)
    summary = {
        'total_rows': len(data['orders']),
        'duplicates': int(
            (rejected['reason'] == 'duplicate').sum()
        ),
        'valid_rows': len(valid),
        'rejected_rows': len(rejected),
        'total_revenue': round(
            float(valid['total'].sum()), 2
        ),
        'files_written': len(written) + 1
    }

    pd.DataFrame(
        list(summary.items()),
        columns=['metric', 'value']
    ).to_csv(
        os.path.join(processed_dir, 'summary.csv'),
        index=False
    )

    return summary

In [0]:
run_pipeline(RAW_DIR, PROCESSED_DIR)

# {'total_rows': 1852,
#  'duplicates': 12,
#  'valid_rows': 1800,
#  'rejected_rows': 52,
#  'total_revenue': 10045400.0,
#  'files_written': 8}

{'total_rows': 1852,
 'duplicates': 12,
 'valid_rows': 1800,
 'rejected_rows': 52,
 'total_revenue': 10045400.0,
 'files_written': 8}